# SPHEREx QR2 spectral mosaic feasibility prototype

This notebook is an **exploratory, bounded experiment**, not a science-ready mosaic pipeline. It keeps spatial discovery, cutout size, detector identity, calibrated wavelength selection, and visualization-only WCS information distinct. Execute sections independently; every section reports missing prerequisites instead of fabricating scientific results.

**Scientific guardrails**
- `IMAGE` is MJy/sr; `VARIANCE` is (MJy/sr)^2. `ZODI` is modeled and is not subtracted.
- LVF wavelength varies across detector position. Geometric channel IDs and `WCS-WAVE` are not calibrated wavelength metadata.
- A science selection requires an explicitly associated full-resolution calibration with `CWAVE`/`CBAND`, a product-tied manifest, detector and calibration identity/version, and matching pixel-grid WCS fingerprints. Equal array shape alone is insufficient.
- If that evidence is unavailable, the fallback remains spatial-only/approximate and **NOT SCIENCE-READY**.
- IRSA's public Mosaic Tool help does not specify its coaddition weights. The custom result is a disclosed unweighted mean; a `reproject_and_coadd` simple-mean output is only a consistency comparison, never an asserted IRSA equivalent.


## A — Configuration and reproducible state

In [1]:
from __future__ import annotations

import hashlib
import json
import re
import resource
import time
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Row, Table
from astropy.utils.data import conf, download_file
from astropy.wcs import WCS
from astroquery.ipac.irsa import Irsa

try:
    from reproject import reproject_interp
    from reproject.mosaicking import find_optimal_celestial_wcs, reproject_and_coadd

    REPROJECT_AVAILABLE = True
    REPROJECT_IMPORT_ERROR = None
except ImportError as exc:
    REPROJECT_AVAILABLE = False
    REPROJECT_IMPORT_ERROR = str(exc)

conf.remote_timeout = 120
RA_DEG = 304.693508808
DEC_DEG = 42.4436872991
SIA_REGION_SIZE_DEG = 0.1
CUTOUT_SIZE_DEG = 0.1
DETECTOR = 2
COLLECTION = "spherex_qr2"
MAX_PRODUCTS = 3
TARGET_CWAVE_UM = None  # Set only for a declared, calibrated selection.
TARGET_CBAND = None
CWAVE_TOLERANCE_UM = None

# Explicitly associate a full-resolution calibration FITS here.
ASSOCIATED_CALIBRATION_PATH = None

# None means reject any non-zero FLAGS value; no bit meanings are inferred.
REJECT_FLAG_BITS = None

if Path("apps/server/data").is_dir():
    DATA_ROOT = Path("apps/server/data")
elif Path("../data").is_dir() and Path.cwd().name == "notebooks":
    DATA_ROOT = Path("../data")
else:
    DATA_ROOT = Path("data")

CACHE_DIR = DATA_ROOT / ".cache/spherex_mosaic"
OUTPUT_DIR = DATA_ROOT / "derived/spherex_mosaic"
ASSOCIATED_CALIBRATION_MANIFEST_PATH: Path | None = (
    DATA_ROOT / "calibration/associated_calibration_manifest.json"
)
COADD_METHOD = "unweighted_mean"
coord = SkyCoord(RA_DEG * u.deg, DEC_DEG * u.deg, frame="icrs")

# Initialize shared state so later sections can run before any network or selection cell.
SIA_REQUIRED_COLUMNS = ("obs_id", "access_url", "energy_bandpassname")
SIA_METADATA_COLUMNS = (
    "obs_id",
    "obs_publisher_did",
    "s_ra",
    "s_dec",
    "s_region",
    "t_min",
    "t_max",
    "energy_bandpassname",
    "em_min",
    "em_max",
    "access_url",
)
SIA_RESULTS = None
SIA_DETECTOR_RESULTS = None
SIA_QUERY_SECONDS = None
SIA_QUERY_ERROR = None
CUTOUTS = []
DOWNLOAD_PROBLEMS = []
SELECTION_PROBLEMS = []
REPROJECTION_PROBLEMS = []
DOWNLOAD_SECONDS = None
REPROJECTED = []
MOSAIC = None
MOSAIC_STATUS = {
    "science_ready": False,
    "label": "NOT SCIENCE-READY",
    "blocker": "Reprojection has not run.",
}
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(
    {
        "ra_deg": RA_DEG,
        "dec_deg": DEC_DEG,
        "sia_region_size_deg": SIA_REGION_SIZE_DEG,
        "cutout_size_deg": CUTOUT_SIZE_DEG,
        "detector": DETECTOR,
        "max_products": MAX_PRODUCTS,
        "reproject_available": REPROJECT_AVAILABLE,
        "reproject_import_error": REPROJECT_IMPORT_ERROR,
        "cache_dir": str(CACHE_DIR),
        "output_dir": str(OUTPUT_DIR),
    }
)

{'ra_deg': 304.693508808, 'dec_deg': 42.4436872991, 'sia_region_size_deg': 0.1, 'cutout_size_deg': 0.1, 'detector': 2, 'max_products': 3, 'reproject_available': True, 'reproject_import_error': None, 'cache_dir': '../data/.cache/spherex_mosaic', 'output_dir': '../data/derived/spherex_mosaic'}


## B — SIA2 discovery with required-column checks

In [ ]:
SIA_REQUIRED_COLUMNS = ("obs_id", "access_url", "energy_bandpassname")
SIA_METADATA_COLUMNS = (
    "obs_id",
    "obs_publisher_did",
    "s_ra",
    "s_dec",
    "s_region",
    "t_min",
    "t_max",
    "energy_bandpassname",
    "em_min",
    "em_max",
    "access_url",
)
SIA_RESULTS = None
SIA_QUERY_SECONDS = None
SIA_QUERY_ERROR = None


def query_products():
    """Query the SIA2 region; detector filtering and product capping happen later."""
    global SIA_RESULTS, SIA_QUERY_SECONDS, SIA_QUERY_ERROR
    started = time.perf_counter()
    try:
        table = Irsa.query_sia(
            pos=(coord, SIA_REGION_SIZE_DEG * u.deg), collection=COLLECTION
        )
        missing = [name for name in SIA_REQUIRED_COLUMNS if name not in table.colnames]
        if missing:
            raise ValueError(f"SIA response is missing essential columns: {missing}")
        SIA_RESULTS = table
        SIA_QUERY_ERROR = None
        absent_metadata = [
            name for name in SIA_METADATA_COLUMNS if name not in table.colnames
        ]
        if absent_metadata:
            print(
                "Optional SIA metadata columns absent and preserved as "
                f"unavailable: {absent_metadata}"
            )
        return table
    except Exception as exc:
        SIA_QUERY_ERROR = f"{type(exc).__name__}: {exc}"
        SIA_RESULTS = None
        print(f"SIA query failed explicitly: {SIA_QUERY_ERROR}")
        return None
    finally:
        SIA_QUERY_SECONDS = time.perf_counter() - started
        print(f"SIA query elapsed: {SIA_QUERY_SECONDS:.3f} s")


if SIA_RESULTS is None:
    query_products()
if SIA_RESULTS is not None:
    print(f"SIA rows: {len(SIA_RESULTS)}; returned columns: {SIA_RESULTS.colnames}")

/home/juand/Documents/vscode/OwnProjects/NasaSpaceApps2026/apps/server/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2611: RuntimeWarning: invalid value encountered in do_format (vectorized)
  outputs = ufunc(*args, out=...)


In [23]:
SIA_RESULTS["access_url"][0]

'https://irsa.ipac.caltech.edu/ibe/data/spherex/qr2/level2/2025W43_1A/l2b-v20-2025-299/2/level2_2025W43_1A_0516_1D2_spx_l2b-v20-2025-299.fits'

In [24]:
SIA_RESULTS.to_pandas()[["obs_id", "energy_bandpassname"]].groupby(
    "energy_bandpassname"
).count().sort_values(by="obs_id", ascending=False)

,obs_id
energy_bandpassname,
SPHEREx-D6,93
SPHEREx-D1,90
SPHEREx-D4,79
SPHEREx-D3,69
SPHEREx-D2,58
SPHEREx-D5,58


## C — Bounded cached cutouts and provenance retention

In [25]:
type(SIA_RESULTS[0])

astropy.table.row.Row

In [26]:
CUTOUTS = []
DOWNLOAD_PROBLEMS = []
DOWNLOAD_SECONDS = 0.0
SIA_DETECTOR_RESULTS = None


def make_cutout_url(access_url: str, center: SkyCoord, size_deg: float) -> str:
    """Add IRSA's documented center/size query to the on-prem access_url."""
    parts = urlsplit(str(access_url))
    query = dict(parse_qsl(parts.query, keep_blank_values=True))
    query.update(
        {"center": f"{center.ra.deg:.10f},{center.dec.deg:.10f}", "size": f"{size_deg:g}"}
    )
    return urlunsplit(
        (parts.scheme, parts.netloc, parts.path, urlencode(query), parts.fragment)
    )


def _table_value(row: Row, name: str) -> str | None:
    value = row[name]
    if np.ma.is_masked(value):
        return None
    if hasattr(value, "item"):
        value = value.item()
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    return str(value)


def detector_from_sia_row(row: Row, colnames: list[str]) -> int | None:
    """Resolve detector from explicit detector metadata or a detector-coded bandpass."""
    for column in ("detector", "spherex_detector"):
        if column in colnames:
            value = _table_value(row, column)
            if value is not None:
                match = re.search(r"([1-6])", str(value))
                if match:
                    return int(match.group(1))
    bandpass = str(_table_value(row, "energy_bandpassname") or "")
    match = re.search(r"(?:^|[^A-Z0-9])D0?([1-6])(?:$|[^0-9])", bandpass.upper())
    return int(match.group(1)) if match else None


def filter_sia_detector(table: Table, detector: int) -> Table | None:
    """Filter before applying MAX_PRODUCTS; fail if SIA rows do not identify a detector"""
    if table is None:
        return None
    keep = []
    unresolved = []
    for index, row in enumerate(table):
        row_detector = detector_from_sia_row(row, table.colnames)
        if row_detector is None:
            unresolved.append(index)
            continue

        if row_detector == int(detector):
            keep.append(index)

    if unresolved:
        raise ValueError(
            f"Cannot determine detector from SIA metadata for rows {unresolved}; "
            "refusing to cap an unfiltered table."
        )

    filtered = table[keep]
    if len(filtered) == 0:
        print(f"No SIA products identify requested detector D{detector}.")

    return filtered


def download_bounded_cutouts(table, max_products=MAX_PRODUCTS):
    """
    Serial, capped download after detector filtering; retain all requested SIA fields.
    """
    cutouts, problems = [], []
    if table is None:
        return cutouts, ["No SIA table is available."]

    detector_rows = filter_sia_detector(table, DETECTOR)
    assert detector_rows is not None, "Detector filtering failed; no rows remain."

    count = min(max(0, int(max_products)), len(detector_rows))
    for index, row in enumerate(detector_rows[:count]):  # pyright: ignore[reportArgumentType]
        provenance = {}
        try:
            provenance = {
                name: str(_table_value(row, name))
                if _table_value(row, name) is not None
                else None
                for name in SIA_METADATA_COLUMNS
                if name in detector_rows.colnames
            }
            url = make_cutout_url(provenance["access_url"], coord, CUTOUT_SIZE_DEG)
            cached_path = download_file(
                url, cache=True, show_progress=False, timeout=conf.remote_timeout
            )
            with fits.open(cached_path, memmap=False, cache=False) as hdul:
                hdul.verify("exception")
                names = {hdu.name.upper(): hdu for hdu in hdul}
                missing = [
                    key for key in ("IMAGE", "FLAGS", "VARIANCE") if key not in names
                ]
                if missing:
                    raise ValueError(f"Cutout missing required HDUs: {missing}")
                image_hdu = names["IMAGE"]
                product_detector = image_hdu.header.get("DETECTOR")

                # Comment from me: Is this check necessary? For me it's redundant because
                # we already filtered the SIA table by detector, but what happens if
                # there's an error by the fits file? The whole fits file is ignored?
                if product_detector is None or int(product_detector) != int(DETECTOR):
                    raise ValueError(
                        f"Product detector {product_detector!r} does not "
                        f"match requested D{DETECTOR}."
                    )

                item = {
                    "index": index,
                    "url": url,
                    "cache_path": str(cached_path),
                    "provenance": provenance,
                    "detector": int(product_detector),
                    "primary_header": hdul[0].header.copy(),
                    "image_header": image_hdu.header.copy(),
                    "variance_header": names["VARIANCE"].header.copy(),
                    "image": np.array(image_hdu.data, dtype=np.float64, copy=True),
                    "flags": np.array(names["FLAGS"].data, copy=True),
                    "variance": np.array(
                        names["VARIANCE"].data, dtype=np.float64, copy=True
                    ),
                    "zodi": (
                        np.array(names["ZODI"].data, dtype=np.float64, copy=True)
                        if "ZODI" in names
                        else None
                    ),
                    "wcs_wave_present": "WCS-WAVE" in names,
                    "hdu_names": [hdu.name for hdu in hdul],
                }
                if not (
                    item["image"].shape == item["variance"].shape == item["flags"].shape
                ):
                    raise ValueError("IMAGE, VARIANCE, and FLAGS shapes differ")
                image_unit = image_hdu.header.get("BUNIT", "").replace(" ", "")
                item["unit_warning"] = (
                    None
                    if image_unit in ("MJy/sr", "MJysr-1")
                    else f"Unexpected IMAGE BUNIT: {image_hdu.header.get('BUNIT')}"
                )
                cutouts.append(item)
        except Exception as exc:
            problem = (
                f"detector-filtered SIA row {index} "
                f"obs_id={provenance.get('obs_id')!r}: {type(exc).__name__}: {exc}"
            )
            problems.append(problem)
            print(f"Cutout failed explicitly: {problem}")
    return cutouts, problems


if SIA_RESULTS is not None and not CUTOUTS:
    _download_started = time.perf_counter()
    try:
        CUTOUTS, DOWNLOAD_PROBLEMS = download_bounded_cutouts(SIA_RESULTS, MAX_PRODUCTS)
        SIA_DETECTOR_RESULTS = filter_sia_detector(SIA_RESULTS, DETECTOR)
    except Exception as exc:
        DOWNLOAD_PROBLEMS.append(
            f"Detector filtering failed: {type(exc).__name__}: {exc}"
        )
        print(DOWNLOAD_PROBLEMS[-1])
    DOWNLOAD_SECONDS = time.perf_counter() - _download_started
print(
    "Detector-filtered SIA rows: "
    f"{len(SIA_DETECTOR_RESULTS) if SIA_DETECTOR_RESULTS is not None else 0}; "
    f"downloaded cutouts: {len(CUTOUTS)} / cap {MAX_PRODUCTS}"
)
print("Download problems:", DOWNLOAD_PROBLEMS)

Detector-filtered SIA rows: 58; downloaded cutouts: 3 / cap 3
Download problems: []


## D — FITS inspection and science-array constraints

In [27]:
def inspect_cutout(item):
    header = item["image_header"]
    image, variance, flags = item["image"], item["variance"], item["flags"]
    spatial_wcs = WCS(header).celestial
    print(
        {
            "index": item["index"],
            "obs_id": item["provenance"].get("obs_id"),
            "detector": item["detector"],
            "shape": image.shape,
            "IMAGE_BUNIT": header.get("BUNIT"),
            "VARIANCE_BUNIT": item["variance_header"].get("BUNIT"),
            "ZODI_present_not_subtracted": item["zodi"] is not None,
            "WCS-WAVE_present_visualization_only": item["wcs_wave_present"],
            "spatial_wcs_valid": bool(spatial_wcs.has_celestial),
            "finite_image": int(np.isfinite(image).sum()),
            "finite_variance_positive": int(
                (np.isfinite(variance) & (variance > 0)).sum()
            ),
            "flagged_pixels": int(np.count_nonzero(flags)),
            "hdu_names": item["hdu_names"],
            "unit_warning": item["unit_warning"],
        }
    )


for _item in CUTOUTS:
    inspect_cutout(_item)

{'index': 0, 'obs_id': '2025W43_1A_0516_1', 'detector': 2, 'shape': (59, 59), 'IMAGE_BUNIT': 'MJy / sr', 'VARIANCE_BUNIT': 'MJy2 / sr2', 'ZODI_present_not_subtracted': True, 'WCS-WAVE_present_visualization_only': True, 'spatial_wcs_valid': True, 'finite_image': 3479, 'finite_variance_positive': 3479, 'flagged_pixels': 3480, 'hdu_names': ['PRIMARY', 'IMAGE', 'FLAGS', 'VARIANCE', 'ZODI', 'PSF', 'WCS-WAVE'], 'unit_warning': None}
{'index': 1, 'obs_id': '2026W20_2A_0191_4', 'detector': 2, 'shape': (59, 59), 'IMAGE_BUNIT': 'MJy / sr', 'VARIANCE_BUNIT': 'MJy2 / sr2', 'ZODI_present_not_subtracted': True, 'WCS-WAVE_present_visualization_only': True, 'spatial_wcs_valid': True, 'finite_image': 3480, 'finite_variance_positive': 3480, 'flagged_pixels': 3481, 'hdu_names': ['PRIMARY', 'IMAGE', 'FLAGS', 'VARIANCE', 'ZODI', 'PSF', 'WCS-WAVE'], 'unit_warning': None}
{'index': 2, 'obs_id': '2026W20_2A_0191_2', 'detector': 2, 'shape': (59, 59), 'IMAGE_BUNIT': 'MJy / sr', 'VARIANCE_BUNIT': 'MJy2 / sr2', '

## E — Calibrated wavelength selection, with an explicit fallback

In [32]:
def celestial_wcs_fingerprint(header):
    """Stable SHA-256 of a celestial WCS header representation."""
    wcs_header = WCS(header).celestial.to_header(relax=True)
    canonical = wcs_header.tostring(sep="\n", endcard=False, padding=False).encode(
        "utf-8"
    )
    return hashlib.sha256(canonical).hexdigest()


def load_associated_calibration(manifest_path, item):
    """
    Require a product-tied manifest, identity, detector, version,
    and proven pixel-grid match.
    """
    if not manifest_path:
        return None, "No per-product Spectral WCS calibration manifest was supplied."
    try:
        manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))
        for key in (
            "calibration_path",
            "calibration_id",
            "calibration_version",
            "detector",
            "product",
            "pixel_alignment",
        ):
            if key not in manifest:
                return None, f"Calibration manifest is missing required field {key!r}."
        source_ids = {
            key: item["provenance"].get(key) for key in ("obs_id", "obs_publisher_did")
        }
        manifest_ids = manifest["product"]
        available_ids = {key: value for key, value in source_ids.items() if value}
        matched_ids = [
            key
            for key, value in available_ids.items()
            if str(manifest_ids.get(key)) == str(value)
        ]
        if not available_ids or len(matched_ids) != len(available_ids):
            return (
                None,
                "Calibration manifest product identity does not match every "
                "available SIA obs_id/obs_publisher_did: "
                f"source={available_ids}, manifest={manifest_ids}.",
            )
        if int(manifest["detector"]) != int(item["detector"]) or int(
            manifest["detector"]
        ) != int(DETECTOR):
            return (
                None,
                f"Calibration manifest detector {manifest['detector']} does not match "
                f"product/requested detector {item['detector']}/{DETECTOR}.",
            )
        if (
            not str(manifest["calibration_id"]).strip()
            or not str(manifest["calibration_version"]).strip()
        ):
            return (
                None,
                "Calibration manifest calibration_id and "
                "calibration_version must be non-empty.",
            )
        alignment = manifest["pixel_alignment"]
        if alignment.get("method") != "identical_celestial_wcs_and_shape":
            return (
                None,
                "Calibration manifest must assert identical_celestial_wcs_and_shape; "
                "shape equality alone is insufficient.",
            )
        science_shape = list(item["image"].shape)
        if (
            alignment.get("science_shape") != science_shape
            or alignment.get("calibration_shape") != science_shape
        ):
            return (
                None,
                "Manifest science/calibration shapes do not exactly match "
                "the product IMAGE shape.",
            )
        science_fingerprint = celestial_wcs_fingerprint(item["image_header"])
        with fits.open(manifest["calibration_path"], memmap=False) as hdul:
            by_name = {hdu.name.upper(): hdu for hdu in hdul}
            if "CWAVE" not in by_name or "CBAND" not in by_name:
                return None, "Calibration product lacks CWAVE and/or CBAND extensions."
            cwave_hdu, cband_hdu = by_name["CWAVE"], by_name["CBAND"]
            cwave = np.asarray(cwave_hdu.data, dtype=float)
            cband = np.asarray(cband_hdu.data, dtype=float)
            if cwave.shape != tuple(science_shape) or cband.shape != tuple(science_shape):
                return (
                    None,
                    f"Calibration data shape mismatch: CWAVE={cwave.shape}, "
                    f"CBAND={cband.shape}, IMAGE={science_shape}.",
                )
            cwave_fingerprint = celestial_wcs_fingerprint(cwave_hdu.header)
            cband_fingerprint = celestial_wcs_fingerprint(cband_hdu.header)
        expected_hash = alignment.get("wcs_sha256")
        if (
            not expected_hash
            or expected_hash != science_fingerprint
            or expected_hash != cwave_fingerprint
            or expected_hash != cband_fingerprint
        ):
            return (
                None,
                "Pixel alignment is unproven: product IMAGE, calibration CWAVE, "
                "calibration CBAND, and manifest WCS SHA-256 must all match.",
            )
        return {
            "cwave": cwave,
            "cband": cband,
            "path": str(manifest["calibration_path"]),
            "manifest_path": str(manifest_path),
            "calibration_id": str(manifest["calibration_id"]),
            "calibration_version": str(manifest["calibration_version"]),
            "product_ids": matched_ids,
            "wcs_sha256": expected_hash,
        }, None
    except Exception as exc:
        return (
            None,
            "Calibration manifest/product validation failed: "
            f"{type(exc).__name__}: {exc}",
        )


def wavelength_selection(
    item,
    calibration,
    target_cwave_um=TARGET_CWAVE_UM,
    target_cband=TARGET_CBAND,
    tolerance_um=CWAVE_TOLERANCE_UM,
):
    """Never promote WCS-WAVE or geometric channel IDs to calibration."""
    if calibration is None:
        return np.ones(item["image"].shape, dtype=bool), {
            "mode": "spatial_only_approximate",
            "science_ready": False,
            "label": "NOT SCIENCE-READY — spatial-only/approximate selection",
            "blocker": (
                "Product-tied calibration manifest with matching detector, "
                "CWAVE/CBAND identity/version, and identical WCS pixel grid "
                "is unavailable or invalid."
            ),
        }
    if target_cwave_um is None and target_cband is None:
        return np.ones(item["image"].shape, dtype=bool), {
            "mode": "calibrated_map_loaded_no_target",
            "science_ready": False,
            "label": "NOT SCIENCE-READY — no wavelength/band selection requested",
            "blocker": (
                "Set a calibrated target CWAVE and/or CBAND before selecting "
                "contributors."
            ),
        }
    cwave, cband = calibration["cwave"], calibration["cband"]
    mask = np.isfinite(cwave) & np.isfinite(cband)
    if target_cwave_um is not None:
        if tolerance_um is None or tolerance_um <= 0:
            return np.zeros(item["image"].shape, dtype=bool), {
                "mode": "blocked",
                "science_ready": False,
                "label": "NOT SCIENCE-READY",
                "blocker": "Positive CWAVE tolerance in micrometers is required.",
            }
        mask &= np.abs(cwave - float(target_cwave_um)) <= float(tolerance_um)
    if target_cband is not None:
        mask &= cband == target_cband
    selected = int(mask.sum())
    return mask, {
        "mode": "calibrated_cwave_cband",
        "science_ready": selected > 0,
        "label": (
            "Calibrated CWAVE/CBAND selection "
            f"({calibration['calibration_id']} "
            f"{calibration['calibration_version']})"
        ),
        "blocker": None
        if selected
        else "No pixels passed the declared CWAVE/CBAND selection.",
        "selected_pixels": selected,
        "cwave_spread_um": [float(np.nanmin(cwave[mask])), float(np.nanmax(cwave[mask]))]
        if selected
        else None,
    }


def valid_science_mask(item, wavelength_mask=None, reject_flag_bits=REJECT_FLAG_BITS):
    mask = (
        np.isfinite(item["image"])
        & np.isfinite(item["variance"])
        & (item["variance"] > 0)
    )
    mask &= (
        (item["flags"] == 0)
        if reject_flag_bits is None
        else ((item["flags"].astype(np.int64) & int(reject_flag_bits)) == 0)
    )
    if wavelength_mask is not None:
        mask &= wavelength_mask
    return mask


SELECTION_PROBLEMS = []
for _item in CUTOUTS:
    try:
        _cal, _blocker = load_associated_calibration(
            ASSOCIATED_CALIBRATION_MANIFEST_PATH, _item
        )
        _wave_mask, _wave_status = wavelength_selection(_item, _cal)
        _item["calibration"] = _cal
        _item["selection_status"] = _wave_status
        _item["valid_mask"] = valid_science_mask(_item, _wave_mask)
        _item["wavelength_selected"] = bool(
            _wave_status.get("science_ready") and np.any(_wave_mask)
        )
        _item["selected_for_quicklook"] = bool(np.any(_item["valid_mask"]))
        if _blocker:
            _item["selection_status"]["blocker"] = _blocker
        print(_item["selection_status"])
    except Exception as exc:
        problem = (
            f"obs_id={_item['provenance'].get('obs_id')!r} selection failed: "
            f"{type(exc).__name__}: {exc}"
        )
        SELECTION_PROBLEMS.append(problem)
        _item["calibration"] = None
        _item["selection_status"] = {
            "mode": "selection_error",
            "science_ready": False,
            "label": "NOT SCIENCE-READY",
            "blocker": problem,
        }
        _item["valid_mask"] = np.zeros(_item["image"].shape, dtype=bool)
        _item["wavelength_selected"] = False
        _item["selected_for_quicklook"] = False
        print(problem)

## F — Celestial reprojection and explicit coaddition policy

In [33]:
REPROJECTED = []
REPROJECTION_PROBLEMS = []
MOSAIC = None
MOSAIC_STATUS = {"science_ready": False, "label": "NOT SCIENCE-READY", "blocker": None}


def spatial_wcs_for(item):
    return WCS(item["image_header"]).celestial


def reproject_contributors(cutouts):
    """Reproject each input independently so a failed product does not abort peers."""
    if not cutouts:
        return [], {
            "science_ready": False,
            "label": "NOT SCIENCE-READY",
            "blocker": "No cutouts were downloaded.",
            "counts": {
                "downloaded": 0,
                "wavelength_selected": 0,
                "selected": 0,
                "reprojected": 0,
                "combined": 0,
            },
        }
    if not REPROJECT_AVAILABLE:
        return [], {
            "science_ready": False,
            "label": "NOT SCIENCE-READY",
            "blocker": (
                f"Required reproject package is unavailable: {REPROJECT_IMPORT_ERROR}"
            ),
            "counts": {
                "downloaded": len(cutouts),
                "wavelength_selected": sum(
                    x.get("wavelength_selected", False) for x in cutouts
                ),
                "selected": sum(x.get("selected_for_quicklook", False) for x in cutouts),
                "reprojected": 0,
                "combined": 0,
            },
        }
    selected = [
        item
        for item in cutouts
        if item.get("selected_for_quicklook")
        and item.get("valid_mask") is not None
        and np.any(item["valid_mask"])
    ]
    if not selected:
        return [], {
            "science_ready": False,
            "label": "NOT SCIENCE-READY",
            "blocker": (
                "No contributor has valid pixels after wavelength/fallback, flag, "
                "variance, and finite-value masks."
            ),
            "counts": {
                "downloaded": len(cutouts),
                "wavelength_selected": sum(
                    x.get("wavelength_selected", False) for x in cutouts
                ),
                "selected": 0,
                "reprojected": 0,
                "combined": 0,
            },
        }
    valid_inputs = []
    for item in selected:
        try:
            wcs = spatial_wcs_for(item)
            if not wcs.has_celestial:
                raise ValueError("IMAGE WCS is not celestial")
            valid_inputs.append((item, wcs))
        except Exception as exc:
            REPROJECTION_PROBLEMS.append(
                f"obs_id={item['provenance'].get('obs_id')!r} WCS validation: "
                f"{type(exc).__name__}: {exc}"
            )
    if not valid_inputs:
        return [], {
            "science_ready": False,
            "label": "NOT SCIENCE-READY",
            "blocker": "No selected contributor has a usable celestial WCS.",
            "counts": {
                "downloaded": len(cutouts),
                "wavelength_selected": sum(
                    x.get("wavelength_selected", False) for x in cutouts
                ),
                "selected": len(selected),
                "reprojected": 0,
                "combined": 0,
            },
        }
    try:
        wcs_out, shape_out = find_optimal_celestial_wcs(
            [(item["image"], wcs) for item, wcs in valid_inputs],
            reference=coord,
            auto_rotate=False,
        )
    except Exception as exc:
        problem = f"Common celestial WCS construction failed: {type(exc).__name__}: {exc}"
        REPROJECTION_PROBLEMS.append(problem)
        return [], {
            "science_ready": False,
            "label": "NOT SCIENCE-READY",
            "blocker": problem,
            "counts": {
                "downloaded": len(cutouts),
                "wavelength_selected": sum(
                    x.get("wavelength_selected", False) for x in cutouts
                ),
                "selected": len(selected),
                "reprojected": 0,
                "combined": 0,
            },
        }
    output = []
    for item, wcs_in in valid_inputs:
        try:
            masked = np.where(item["valid_mask"], item["image"], np.nan)
            reproj_image, footprint = reproject_interp(
                (masked, wcs_in),
                wcs_out,
                shape_out=shape_out,
                order="bilinear",
                return_footprint=True,
            )
            reproj_variance, var_footprint = reproject_interp(
                (np.where(item["valid_mask"], item["variance"], np.nan), wcs_in),
                wcs_out,
                shape_out=shape_out,
                order="bilinear",
                return_footprint=True,
            )
            yy, xx = np.indices(shape_out, dtype=float)
            world = wcs_out.pixel_to_world(xx, yy)
            sx, sy = wcs_in.world_to_pixel(world)
            sxn, syn = np.rint(sx).astype(int), np.rint(sy).astype(int)
            inside = (
                (sxn >= 0)
                & (syn >= 0)
                & (sxn < item["flags"].shape[1])
                & (syn < item["flags"].shape[0])
            )
            flags = np.zeros(shape_out, dtype=item["flags"].dtype)
            flags[inside] = item["flags"][syn[inside], sxn[inside]]
            usable = (
                (footprint > 0)
                & (var_footprint > 0)
                & np.isfinite(reproj_image)
                & np.isfinite(reproj_variance)
                & (reproj_variance > 0)
            )
            if not np.any(usable):
                raise ValueError("No valid output pixels after reprojection")
            output.append(
                {
                    "source": item,
                    "image": reproj_image,
                    "variance": reproj_variance,
                    "flags": flags,
                    "coverage": usable.astype(np.uint16),
                    "wcs": wcs_out,
                    "valid": usable,
                }
            )
        except Exception as exc:
            problem = (
                f"obs_id={item['provenance'].get('obs_id')!r} reprojection failed: "
                f"{type(exc).__name__}: {exc}"
            )
            REPROJECTION_PROBLEMS.append(problem)
            print(problem)
    calibrated = bool(output) and all(
        x["source"].get("wavelength_selected", False) for x in output
    )
    counts = {
        "downloaded": len(cutouts),
        "wavelength_selected": sum(x.get("wavelength_selected", False) for x in cutouts),
        "selected": len(selected),
        "reprojected": len(output),
        "combined": len(output) if len(output) >= 2 else 0,
    }
    if not output:
        label = "NOT SCIENCE-READY"
        blocker = "No contributor was successfully reprojected."
    elif len(output) == 1:
        label = (
            "NOT SCIENCE-READY — single-contributor reprojected quick-look (not a mosaic)"
        )
        blocker = (
            "Only one contributor was reprojected; a single input is not a "
            "multi-exposure mosaic."
        )
    elif not calibrated:
        label = (
            "NOT SCIENCE-READY — spatial-only/approximate multi-contributor quick-look"
        )
        blocker = (
            "At least one contributor lacks validated product-associated "
            "CWAVE/CBAND selection."
        )
    else:
        label = "Calibrated multi-exposure exploratory mosaic"
        blocker = None
    return output, {
        "science_ready": calibrated and len(output) >= 2,
        "label": label,
        "blocker": blocker,
        "counts": counts,
        "shape": list(shape_out),
        "wcs": wcs_out,
        "problems": list(REPROJECTION_PROBLEMS),
    }


REPROJECTED, MOSAIC_STATUS = reproject_contributors(CUTOUTS)
if REPROJECTED:
    stack = np.stack(
        [np.where(item["valid"], item["image"], np.nan) for item in REPROJECTED]
    )
    coverage = np.sum(
        np.stack([item["valid"] for item in REPROJECTED]), axis=0, dtype=np.uint16
    )
    if len(REPROJECTED) >= 2:
        image_sum = np.nansum(stack, axis=0)
        mosaic_image = np.divide(
            image_sum,
            coverage,
            out=np.full(coverage.shape, np.nan, dtype=np.float64),
            where=coverage > 0,
        )
        variance_stack = np.stack(
            [np.where(item["valid"], item["variance"], np.nan) for item in REPROJECTED]
        )
        mosaic_variance = np.nansum(variance_stack, axis=0) / np.maximum(coverage, 1) ** 2
        combination = (
            "custom unweighted nanmean of valid reprojected IMAGE values; "
            "VARIANCE approximated as sum(input VARIANCE)/N^2; "
            "no inverse-variance weighting"
        )
    else:
        mosaic_image = np.where(REPROJECTED[0]["valid"], REPROJECTED[0]["image"], np.nan)
        mosaic_variance = np.where(
            REPROJECTED[0]["valid"], REPROJECTED[0]["variance"], np.nan
        )
        combination = "single contributor passthrough; no coaddition and not a mosaic"
    mosaic_variance[coverage == 0] = np.nan
    mosaic_flags = np.bitwise_or.reduce(
        np.stack([item["flags"] for item in REPROJECTED]), axis=0
    )
    wavelength_samples = []
    for contributor in REPROJECTED:
        source = contributor["source"]
        calibration = source.get("calibration")
        if calibration is not None:
            valid = source["valid_mask"] & np.isfinite(calibration["cwave"])
            if valid.any():
                wavelength_samples.append(calibration["cwave"][valid])
    wavelength_spread = None
    if wavelength_samples:
        all_cwave = np.concatenate(wavelength_samples)
        wavelength_spread = {
            "min_um": float(np.nanmin(all_cwave)),
            "max_um": float(np.nanmax(all_cwave)),
            "span_um": float(np.nanmax(all_cwave) - np.nanmin(all_cwave)),
        }
    MOSAIC = {
        "image": mosaic_image,
        "variance": mosaic_variance,
        "coverage": coverage,
        "nhits": coverage.copy(),
        "flags": mosaic_flags,
        "wcs": REPROJECTED[0]["wcs"],
        "combination": combination,
        "wavelength_spread_um": wavelength_spread,
        "simple_mean_comparison": None,
    }
    if len(REPROJECTED) >= 2:
        comparison_inputs = [
            (
                np.where(item["source"]["valid_mask"], item["source"]["image"], np.nan),
                spatial_wcs_for(item["source"]),
            )
            for item in REPROJECTED
        ]
        try:
            comparison, comparison_footprint = reproject_and_coadd(
                comparison_inputs,
                MOSAIC["wcs"],
                shape_out=MOSAIC["image"].shape,
                reproject_function=reproject_interp,
                combine_function="mean",
            )
            comparable = (
                (MOSAIC["coverage"] > 0)
                & (comparison_footprint > 0)
                & np.isfinite(comparison)
                & np.isfinite(MOSAIC["image"])
            )
            MOSAIC["simple_mean_comparison"] = {
                "label": (
                    "reproject_and_coadd simple-mean consistency comparison; "
                    "not IRSA-equivalent"
                ),
                "valid_pixels": int(comparable.sum()),
                "mean_absolute_difference": float(
                    np.nanmean(
                        np.abs(comparison[comparable] - MOSAIC["image"][comparable])
                    )
                )
                if comparable.any()
                else None,
                "max_absolute_difference": float(
                    np.nanmax(
                        np.abs(comparison[comparable] - MOSAIC["image"][comparable])
                    )
                )
                if comparable.any()
                else None,
            }
        except Exception as exc:
            problem = (
                "reproject_and_coadd consistency comparison failed: "
                f"{type(exc).__name__}: {exc}"
            )
            REPROJECTION_PROBLEMS.append(problem)
            MOSAIC["simple_mean_comparison"] = {
                "label": "comparison unavailable; not IRSA-equivalent",
                "error": problem,
            }
    print({k: v for k, v in MOSAIC_STATUS.items() if k != "wcs"})
    print(MOSAIC["combination"])
    print(MOSAIC["simple_mean_comparison"])
else:
    print(MOSAIC_STATUS)

{'science_ready': False, 'label': 'NOT SCIENCE-READY', 'blocker': 'No cutouts were downloaded.', 'counts': {'downloaded': 0, 'wavelength_selected': 0, 'selected': 0, 'reprojected': 0, 'combined': 0}}


## G — Individual, selected, reprojected, final, and coverage plots

In [ ]:
def plot_array(data, title, cbar_label, cmap="gray_r"):
    finite = np.isfinite(data)
    fig, ax = plt.subplots(figsize=(7, 6))
    if finite.any():
        lo, hi = np.nanpercentile(data[finite], [1, 99])
        if lo == hi:
            hi = lo + 1
        artist = ax.imshow(data, origin="lower", cmap=cmap, vmin=lo, vmax=hi)
        fig.colorbar(artist, ax=ax, label=cbar_label)
    else:
        ax.text(
            0.5, 0.5, "No finite pixels", ha="center", va="center", transform=ax.transAxes
        )
    ax.set_title(title)
    ax.set_xlabel("Pixel X")
    ax.set_ylabel("Pixel Y")
    fig.tight_layout()
    return fig


for item in CUTOUTS:
    plot_array(
        item["image"],
        f"Individual IMAGE — obs_id={item['provenance'].get('obs_id')}",
        "MJy/sr",
    )
    selected = np.where(
        item.get("valid_mask", np.zeros(item["image"].shape, dtype=bool)),
        item["image"],
        np.nan,
    )
    plot_array(
        selected, f"Selected IMAGE — {item['selection_status']['label']}", "MJy/sr"
    )
for item in REPROJECTED:
    plot_array(
        item["image"],
        f"Reprojected — obs_id={item['source']['provenance'].get('obs_id')}",
        "MJy/sr",
    )
if MOSAIC is not None:
    plot_array(MOSAIC["image"], f"Final product — {MOSAIC_STATUS['label']}", "MJy/sr")
    plot_array(
        MOSAIC["coverage"], "Coverage / NHITS", "contributing cutouts", cmap="viridis"
    )

## H — FITS, PNG, JSON provenance, and coverage export

In [ ]:
def metadata_document(cutouts, mosaic, status, problems):
    records = []
    for item in cutouts:
        record = {name: item["provenance"].get(name) for name in SIA_METADATA_COLUMNS}
        record.update(
            {
                "detector": int(item["detector"]),
                "shape": list(item["image"].shape),
                "image_unit": item["image_header"].get("BUNIT"),
                "variance_unit": item["variance_header"].get("BUNIT"),
                "zodi_subtracted": False,
                "wcs_wave_used_for_selection": False,
                "calibration_path": (item.get("calibration") or {}).get("path"),
                "calibration_id": (item.get("calibration") or {}).get("calibration_id"),
                "calibration_version": (item.get("calibration") or {}).get(
                    "calibration_version"
                ),
                "selection": item.get("selection_status"),
                "wavelength_selected": bool(item.get("wavelength_selected", False)),
                "selected_for_quicklook": bool(item.get("selected_for_quicklook", False)),
                "reprojected": any(result["source"] is item for result in REPROJECTED),
                "flag_policy": "reject nonzero flags"
                if REJECT_FLAG_BITS is None
                else f"reject bits in mask {REJECT_FLAG_BITS}",
                "version": item["primary_header"].get("VERSION"),
                "hdu_names": item["hdu_names"],
            }
        )
        records.append(record)
    counts = dict(MOSAIC_STATUS.get("counts", {}))
    counts.setdefault("downloaded", len(cutouts))
    counts.setdefault(
        "wavelength_selected", sum(x.get("wavelength_selected", False) for x in cutouts)
    )
    counts.setdefault(
        "selected", sum(x.get("selected_for_quicklook", False) for x in cutouts)
    )
    counts.setdefault("reprojected", len(REPROJECTED))
    counts.setdefault("combined", len(REPROJECTED) if len(REPROJECTED) >= 2 else 0)
    return {
        "schema": "spherex-mosaic-prototype/v2",
        "science_ready": bool(status.get("science_ready", False)),
        "label": status.get("label", "NOT SCIENCE-READY"),
        "blocker": status.get("blocker"),
        "counts": counts,
        "configuration": {
            "ra_deg": RA_DEG,
            "dec_deg": DEC_DEG,
            "sia_region_size_deg": SIA_REGION_SIZE_DEG,
            "cutout_size_deg": CUTOUT_SIZE_DEG,
            "detector": DETECTOR,
            "max_products": MAX_PRODUCTS,
            "target_cwave_um": TARGET_CWAVE_UM,
            "target_cband": TARGET_CBAND,
            "cwave_tolerance_um": CWAVE_TOLERANCE_UM,
        },
        "query": {
            "collection": COLLECTION,
            "essential_columns": list(SIA_REQUIRED_COLUMNS),
            "metadata_columns": list(SIA_METADATA_COLUMNS),
            "elapsed_seconds": SIA_QUERY_SECONDS,
            "error": SIA_QUERY_ERROR,
        },
        "contributors": records,
        "combination": mosaic.get("combination") if mosaic else None,
        "simple_mean_comparison": mosaic.get("simple_mean_comparison")
        if mosaic
        else None,
        "variance_weighting": False,
        "coverage_definition": (
            "NHITS = number of successfully reprojected valid contributors "
            "per output pixel"
        ),
        "wavelength_spread_um": mosaic.get("wavelength_spread_um") if mosaic else None,
        "problems": list(problems)
        + list(SELECTION_PROBLEMS)
        + list(REPROJECTION_PROBLEMS),
    }


def export_products(cutouts, mosaic, status, problems, output_dir=OUTPUT_DIR):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    meta = metadata_document(cutouts, mosaic, status, problems)
    json_path = output_dir / "mosaic_metadata.json"
    json_path.write_text(json.dumps(meta, indent=2, default=str) + "\n", encoding="utf-8")
    files = {"metadata_json": str(json_path)}
    if mosaic is None:
        return files, meta
    image_unit = (u.MJy / u.sr).to_string(format="fits")
    variance_unit = ((u.MJy / u.sr) ** 2).to_string(format="fits")
    wcs_header = mosaic["wcs"].to_header()
    image_header = wcs_header.copy()
    image_header["BUNIT"] = image_unit
    image_header["COMBMETH"] = (
        "UNWEIGHTED MEAN" if len(REPROJECTED) >= 2 else "PASSTHROUGH"
    )
    variance_header = wcs_header.copy()
    variance_header["BUNIT"] = variance_unit
    count_header = wcs_header.copy()
    count_header["BUNIT"] = "1"
    count_header["COMMENT"] = "NHITS is a dimensionless integer contributor count."
    flags_header = wcs_header.copy()
    flags_header["BUNIT"] = "1"
    flags_header["COMMENT"] = (
        "FLAGS is a discrete integer bitmask; not interpolated as continuous data."
    )
    hdus = [
        fits.PrimaryHDU(),
        fits.ImageHDU(mosaic["image"], header=image_header, name="IMAGE"),
        fits.ImageHDU(mosaic["variance"], header=variance_header, name="VARIANCE"),
        fits.ImageHDU(mosaic["coverage"], header=count_header, name="NHITS"),
        fits.ImageHDU(mosaic["flags"], header=flags_header, name="FLAGS"),
    ]
    fits_path = output_dir / "spherex_mosaic_prototype.fits"
    fits.HDUList(hdus).writeto(fits_path, overwrite=True, checksum=True)
    files["fits"] = str(fits_path)
    for name, array, label in [
        ("mosaic", mosaic["image"], status.get("label", "Quick-look")),
        ("coverage", mosaic["coverage"], "Coverage / NHITS"),
    ]:
        fig = plot_array(
            array,
            label,
            "MJy/sr" if name == "mosaic" else "contributing cutouts",
            cmap="gray_r" if name == "mosaic" else "viridis",
        )
        png_path = output_dir / f"{name}.png"
        fig.savefig(png_path, dpi=140)
        plt.close(fig)
        files[f"{name}_png"] = str(png_path)
    return files, meta


EXPORT_FILES, EXPORT_METADATA = export_products(
    CUTOUTS,
    MOSAIC,
    MOSAIC_STATUS,
    (DOWNLOAD_PROBLEMS or [])
    + (SELECTION_PROBLEMS or [])
    + (REPROJECTION_PROBLEMS or []),
)
print(EXPORT_FILES)
print(
    {
        k: EXPORT_METADATA[k]
        for k in ("schema", "science_ready", "label", "blocker", "counts")
    }
)

## I — Evaluation: timings, counts, memory, problems, backend recommendations

In [ ]:
def memory_peak_mb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024


counts = (
    EXPORT_METADATA["counts"]
    if "EXPORT_METADATA" in globals()
    else MOSAIC_STATUS.get("counts", {})
)
EVALUATION = {
    "sia_query_seconds": SIA_QUERY_SECONDS,
    "download_seconds": DOWNLOAD_SECONDS if DOWNLOAD_SECONDS else None,
    "sia_rows": len(SIA_RESULTS) if SIA_RESULTS is not None else 0,
    "detector_sia_rows": len(SIA_DETECTOR_RESULTS)
    if SIA_DETECTOR_RESULTS is not None
    else 0,
    "downloaded_count": counts.get("downloaded", len(CUTOUTS)),
    "wavelength_selected_count": counts.get(
        "wavelength_selected", sum(x.get("wavelength_selected", False) for x in CUTOUTS)
    ),
    "selected_count": counts.get(
        "selected", sum(x.get("selected_for_quicklook", False) for x in CUTOUTS)
    ),
    "reprojected_count": counts.get("reprojected", len(REPROJECTED)),
    "combined_count": counts.get(
        "combined", len(REPROJECTED) if len(REPROJECTED) >= 2 else 0
    ),
    "multi_exposure_mosaic": bool(len(REPROJECTED) >= 2),
    "cutout_cap": MAX_PRODUCTS,
    "output_shape": list(MOSAIC["image"].shape) if MOSAIC else None,
    "coverage_pixels": int(np.count_nonzero(MOSAIC["coverage"])) if MOSAIC else 0,
    "peak_process_memory_mb_approx": memory_peak_mb(),
    "problems": list(DOWNLOAD_PROBLEMS)
    + list(SELECTION_PROBLEMS)
    + list(REPROJECTION_PROBLEMS)
    + ([SIA_QUERY_ERROR] if SIA_QUERY_ERROR else []),
    "backend_recommendations": [
        "Filter detector-coded SIA rows before limiting downloads.",
        "Require a product-specific calibration manifest and WCS fingerprint "
        "before science selection.",
        "Keep FLAGS discrete and preserve per-extension units and provenance.",
        "Benchmark reprojection/memory on multiple associated exposures.",
        "The simple-mean comparison is not asserted equivalent to IRSA weighting.",
    ],
}
print(json.dumps(EVALUATION, indent=2, default=str))

## J — Interpretation and limitations


### Current interpretation

- SIA discovery, requested cutout detector, calibrated wavelength selection, and usable reprojection are separate counts. Evaluation reports downloaded, wavelength-selected, quick-look-selected, reprojected, and combined contributors independently.
- Calibration is accepted only with a per-product JSON manifest that matches `obs_id` and/or `obs_publisher_did`, the requested detector, a non-empty calibration identity/version, exact image/CWAVE/CBAND shapes, and matching SHA-256 fingerprints of their celestial WCS headers. The manifest must point to a calibration FITS with `CWAVE` and `CBAND`. Shape equality alone is explicitly insufficient.
- If the required manifest or WCS evidence is missing, selection falls back to spatial-only/approximate and is **NOT SCIENCE-READY**. No `WCS-WAVE` or geometric channel ID is promoted to calibration.
- A single reprojected contributor is a passthrough quick-look, not a mosaic. Multi-exposure status requires at least two successful reprojections; science-ready additionally requires calibrated selection for every combined contributor.
- `reproject_and_coadd` is run only for at least two successful contributors as a simple-mean consistency comparison. The custom output retains masks, variance approximation, bit flags, coverage, and provenance; the comparison is not assumed equivalent to the custom result or IRSA's proprietary weighting.
- `IMAGE` and `VARIANCE` are exported with independent FITS-valid units. `NHITS` and `FLAGS` are integer/dimensionless; flags are sampled as discrete bitmasks and never interpolated as intensities.
- Bilinear interpolation changes noise covariance; the accompanying variance remains approximate and is not inverse-variance weighted. ZODI is not subtracted.

### Required calibration manifest

Provide a JSON manifest per SIA product with `calibration_path`, non-empty `calibration_id` and `calibration_version`, integer `detector`, `product` containing matching `obs_id` and/or `obs_publisher_did`, and `pixel_alignment` containing method `identical_celestial_wcs_and_shape`, the exact science/calibration array shapes, and `wcs_sha256`. The WCS hash must match the notebook-computed celestial WCS fingerprint for IMAGE, CWAVE, and CBAND. Otherwise the recorded blocker remains: product association, detector, calibration identity/version, or pixel alignment is unproven.

### Remaining science validation

Run with at least two real compatible exposures and the actual associated calibration manifests; validate calibration provenance, flux behavior, variance/covariance, flag propagation, wavelength spread, and product counts. A synthetic offline two-input smoke test validates software paths only; it is not scientific evidence. Do not call the fallback a science mosaic.
